In [1]:
# --- imports ---
import pandas as pd
import sqlite3

print("All imports OK")

All imports OK


In [2]:
# --- config ---
MODE = "streetcar"
DATASET = "../data/ttc-streetcar-delay-data-2025.csv"
LOOKUP = "../data/ttc-streetcar-delay-data-lookup.csv"
DB = "../sql/TTC_delay.db"

In [3]:
df = pd.read_csv(DATASET)
df.head()

,_id,Date,Line,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Vehicle
0,1,2025-01-01,504 KING,02:10,Wednesday,KING AND PARLIAMENT,MTSAN,10,20,W,4569
1,2,2025-01-01,506 CARLTON,02:50,Wednesday,COLLEGE AND HURON,MTAFR,34,49,E,4480
2,3,2025-01-01,504 KING,03:11,Wednesday,KING AND QUEEN EAST,MTIE,15,30,E,4629
3,4,2025-01-01,501 QUEEN,05:14,Wednesday,QUEEN AND DUFFERIN,MTAFR,36,55,E,4455
4,5,2025-01-01,504 KING,06:14,Wednesday,LESLIE CARHOUSE,MTVIS,10,20,E,4483


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23703 entries, 0 to 23702
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   _id        23703 non-null  int64 
 1   Date       23703 non-null  object
 2   Line       23351 non-null  object
 3   Time       23703 non-null  object
 4   Day        23703 non-null  object
 5   Station    23703 non-null  object
 6   Code       23702 non-null  object
 7   Min Delay  23703 non-null  int64 
 8   Min Gap    23703 non-null  int64 
 9   Bound      18376 non-null  object
 10  Vehicle    23703 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 2.0+ MB


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isna().sum()

_id             0
Date            0
Line          352
Time            0
Day             0
Station         0
Code            1
Min Delay       0
Min Gap         0
Bound        5327
Vehicle         0
dtype: int64

In [7]:
# catching stray values
df["Bound"].value_counts(dropna=False)

Bound
W      6874
E      6785
NaN    5327
S      2361
N      2339
B        16
8         1
Name: count, dtype: int64

In [8]:
valid_directions = ['N', 'S', 'E', 'W']

#  group anything that is NOT a valid direction (including NaNs) and set it to "Unknown"
df.loc[~df['Bound'].isin(valid_directions), 'Bound'] = 'Unknown'

# recheck
df["Bound"].value_counts(dropna=False)

Bound
W          6874
E          6785
Unknown    5344
S          2361
N          2339
Name: count, dtype: int64

In [9]:
# standardize column names
df = df.rename(columns={"_id": "source_id",
                        "Min Delay": "min_delay", "Min Gap": "min_gap",})
df.columns = df.columns.str.lower()

# keep missing streetcar line and code
df["line"] = df["line"].fillna("Unknown") 
df["code"] = df["code"].fillna("Uncoded") 

# time format
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
df["quarter"]  = df["date"].dt.year.astype(str) + "-Q" + df["date"].dt.quarter.astype(str)
df["hour"]  = df["time"].str[:2].astype(int)

# new identifiers
df["is_delay"] = (df["min_delay"] > 0).astype(int)
df["source_id"] = MODE + "-" + df["source_id"].astype(str)


df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23703 entries, 0 to 23702
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   source_id  23703 non-null  object        
 1   date       23703 non-null  datetime64[ns]
 2   line       23703 non-null  object        
 3   time       23703 non-null  object        
 4   day        23703 non-null  object        
 5   station    23703 non-null  object        
 6   code       23703 non-null  object        
 7   min_delay  23703 non-null  int64         
 8   min_gap    23703 non-null  int64         
 9   bound      23703 non-null  object        
 10  vehicle    23703 non-null  int64         
 11  month      23703 non-null  object        
 12  quarter    23703 non-null  object        
 13  hour       23703 non-null  int64         
 14  is_delay   23703 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(9)
memory usage: 2.7+ MB


In [10]:
# load lookup table and do basic cleaning
lookup = pd.read_csv(LOOKUP, encoding="utf-8")
lookup.columns = lookup.columns.str.lower()
lookup = lookup[["code", "description"]].drop_duplicates("code")

# join table on "code"
df = df.merge(lookup, on="code", how="left")
df["description"] = df["description"].fillna("Unknown")

df.head()

,source_id,date,line,time,day,station,code,min_delay,min_gap,bound,vehicle,month,quarter,hour,is_delay,description
0,streetcar-1,2025-01-01,504 KING,02:10,Wednesday,KING AND PARLIAMENT,MTSAN,10,20,W,4569,2025-01,2025-Q1,2,1,UNSANITARY VEHICLE
1,streetcar-2,2025-01-01,506 CARLTON,02:50,Wednesday,COLLEGE AND HURON,MTAFR,34,49,E,4480,2025-01,2025-Q1,2,1,AUTO FOUL RAIL
2,streetcar-3,2025-01-01,504 KING,03:11,Wednesday,KING AND QUEEN EAST,MTIE,15,30,E,4629,2025-01,2025-Q1,3,1,INJURED EMPLOYEES
3,streetcar-4,2025-01-01,501 QUEEN,05:14,Wednesday,QUEEN AND DUFFERIN,MTAFR,36,55,E,4455,2025-01,2025-Q1,5,1,AUTO FOUL RAIL
4,streetcar-5,2025-01-01,504 KING,06:14,Wednesday,LESLIE CARHOUSE,MTVIS,10,20,E,4483,2025-01,2025-Q1,6,1,VISION ISSUES


In [11]:
df.tail()

,source_id,date,line,time,day,station,code,min_delay,min_gap,bound,vehicle,month,quarter,hour,is_delay,description
23698,streetcar-23699,2026-08-31,507 LONG BRANCH,19:30,Monday,3765 LAKE SHORE BLVD W,EFO,10,20,W,3213,2026-08,2026-Q3,19,1,Unknown
23699,streetcar-23700,2026-08-31,505 DUNDAS,22:29,Monday,DUNDAS AND JARVIS,TTPD,6,15,E,4652,2026-08,2026-Q3,22,1,TTC COLLISION â PROPERTY DAMAGE
23700,streetcar-23701,2026-08-31,505 DUNDAS,23:50,Monday,BROADVIEW STATION,MTSAN,10,20,E,4500,2026-08,2026-Q3,23,1,UNSANITARY VEHICLE
23701,streetcar-23702,2026-08-31,506 CARLTON,00:05,Monday,MAIN STREET STATION,MTBIO,10,20,W,4650,2026-08,2026-Q3,0,1,Unknown
23702,streetcar-23703,2026-08-31,504 KING,00:43,Monday,KING AND PORTLAND,MTAFR,14,24,E,4445,2026-08,2026-Q3,0,1,AUTO FOUL RAIL


In [12]:
# sqlite
conn = sqlite3.connect(DB)
df.to_sql(f"{MODE}_delays", conn, if_exists="replace", index=False)

groups = {"month": "month", "hour": "hour", "reason": "description"}
for name, col in groups.items():
    conn.execute(f"DROP VIEW IF EXISTS view_{MODE}_by_{name}")
    conn.execute(f"""
    CREATE VIEW view_{MODE}_by_{name} AS
    SELECT {col},
           COUNT(*) AS incidents,
           SUM(min_delay) AS total_delay_min,
           ROUND(AVG(CASE WHEN is_delay = 1 THEN min_delay END), 2) AS avg_delay_min
    FROM {MODE}_delays
    GROUP BY {col}""")
conn.commit()

print(f"{MODE} view created successfully.")
conn.close()

streetcar view created successfully.
